# Chapter 5: RAG and File Tools


In [ ]:
from dotenv import load_dotenv
load_dotenv()

%load_ext autoreload
%autoreload 2


## 5.1 Embeddings


In [ ]:
from scratch_agent.rag import get_embeddings
import numpy as np

# Get embeddings for sample texts
texts = ["The cat sat on the mat", "A dog played in the park", "The cat rested on the rug"]
embeddings = get_embeddings(texts)

# Compute cosine similarity
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"Similarity (cat/cat): {cosine_similarity(embeddings[0], embeddings[2]):.4f}")
print(f"Similarity (cat/dog): {cosine_similarity(embeddings[0], embeddings[1]):.4f}")


## 5.2 Chunking


In [ ]:
from scratch_agent.rag import fixed_length_chunking

# Demo fixed-length chunking
long_text = "This is a sample document. " * 50
chunks = fixed_length_chunking(long_text)

print(f"Original length: {len(long_text)} characters")
print(f"Number of chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i}: {len(chunk)} chars - {chunk[:60]}...")


## 5.3 Vector Search


In [ ]:
from scratch_agent.rag import get_embeddings, vector_search

# Demo vector search with sample documents
documents = [
    "Python is a programming language known for its simplicity.",
    "Machine learning is a subset of artificial intelligence.",
    "The Eiffel Tower is located in Paris, France.",
    "Neural networks are inspired by biological brain structures.",
    "The Great Wall of China is visible from space.",
]

# First, get embeddings for documents
doc_embeddings = get_embeddings(documents)

query = "What is deep learning?"
results = vector_search(query, documents, doc_embeddings)
print(f"Query: {query}")
for r in results:
    print(f"  [{r['similarity']:.4f}] {r['chunk']}")

## 5.4 RAG Pipeline


In [ ]:
from scratch_agent.rag import get_embeddings, fixed_length_chunking, vector_search

# Full RAG pipeline: chunk -> embed -> query
raw_text = """Artificial intelligence (AI) is intelligence demonstrated by machines.
Machine learning is a subset of AI that enables systems to learn from data.
Deep learning uses neural networks with many layers to model complex patterns.
Natural language processing (NLP) helps computers understand human language.
Computer vision allows machines to interpret and understand visual information."""

# Step 1: Chunk the text
chunks = fixed_length_chunking(raw_text, chunk_size=100, overlap=20)
print(f"Created {len(chunks)} chunks")

# Step 2: Embed the chunks
chunk_embeddings = get_embeddings(chunks)

# Step 3: Search with a query
query = "How do computers understand language?"
results = vector_search(query, chunks, chunk_embeddings)
print(f"\nQuery: {query}")
for r in results:
    print(f"  [{r['similarity']:.4f}] {r['chunk']}")

## 5.5 File Tools


In [ ]:
from scratch_agent.tools.file_tools import unzip_file, list_files, read_file, read_media_file
from scratch_agent.tools.base import FunctionTool

# Wrap file tools as FunctionTools for use with the agent
file_tools = [
    FunctionTool(func=unzip_file),
    FunctionTool(func=list_files),
    FunctionTool(func=read_file),
    FunctionTool(func=read_media_file),
]

for ft in file_tools:
    print(f"{ft.name}: {ft.description[:80]}")
    print()

## 5.6 Callbacks


In [ ]:
from scratch_agent.callbacks import approval_callback, search_compressor
from scratch_agent.agent import Agent
from scratch_agent.llm import LlmClient
from scratch_agent.tools.base import tool

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

# Create an agent with before/after tool callbacks
agent = Agent(
    model=LlmClient(model="gpt-4o-mini"),
    tools=[calculator],
    instructions="You are a helpful assistant.",
    before_tool_callbacks=[approval_callback],
    after_tool_callbacks=[search_compressor],
)
print(f"Agent created with callbacks")
print(f"Before-tool callbacks: {[cb.__name__ for cb in agent.before_tool_callbacks]}")
print(f"After-tool callbacks: {[cb.__name__ for cb in agent.after_tool_callbacks]}")

## 5.7 Agent with RAG and File Tools


In [ ]:
from scratch_agent.agent import Agent
from scratch_agent.llm import LlmClient
from scratch_agent.tools.base import FunctionTool
from scratch_agent.tools.file_tools import list_files, read_file

# Create agent with file tools
file_agent = Agent(
    model=LlmClient(model="gpt-4o-mini"),
    tools=[FunctionTool(func=list_files), FunctionTool(func=read_file)],
    instructions="You are a helpful assistant that can read and process files.",
    max_steps=5,
)

# Run on a sample task
result = await file_agent.run("List the files in the current directory.")
print(result.output)